# BCRA — Web Scraping vs API REST

Este notebook compara **dos formas de obtener los mismos datos** del Banco Central de la República Argentina (BCRA): tipo de cambio minorista (USD), **Serie 246**.

| Enfoque | Cómo funciona | Cuándo usarlo |
|---------|---------------|---------------|
| **Web Scraping** | Descarga el HTML de la página y extrae los datos del DOM | Cuando no hay API disponible |
| **API REST** | Llama directamente al endpoint JSON oficial del BCRA | Siempre que exista — más limpio y estable |

**URLs utilizadas:**
- Página web: `https://www.bcra.gob.ar/principales-variables-resultados/`
- API oficial: `https://api.bcra.gob.ar/estadisticas/v3/monetarias/246`

**Tecnologías:** `requests`, `beautifulsoup4`, `pandas`, `pyarrow`

## 1. Instalación de Dependencias

In [ ]:

%pip install requests beautifulsoup4 pandas pyarrow lxml curl_cffi fake-useragent


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importación de Librerías y Parámetros

In [1]:
import requests
import pandas as pd
import warnings
from bs4 import BeautifulSoup
from pathlib import Path

# Suprimir advertencias de SSL (el BCRA usa certificado propio)
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

DIR_OUTPUT = Path('datos/output')
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

# Rango de fechas a consultar
FECHA_DESDE = '2025-04-15'
FECHA_HASTA = '2026-04-15'
SERIE       = 246   # Tipo de Cambio Minorista ($ por USD) — Com. B 9791

print('Librerías cargadas.')
print(f'Período: {FECHA_DESDE} → {FECHA_HASTA}')
print(f'Serie BCRA: {SERIE}')

Librerías cargadas.
Período: 2025-04-15 → 2026-04-15
Serie BCRA: 246


---
## PARTE A — Enfoque 1: Web Scraping

Descargamos el HTML de la página de resultados del BCRA y buscamos los datos en el DOM.

### A1. Petición GET y HTML Crudo

In [3]:

from fake_useragent import UserAgent
from curl_cffi import requests as curl_requests

url_web = (
    f'https://www.bcra.gob.ar/principales-variables-resultados/'
    f'?serie={SERIE}&serie1=0&serie2=0&serie3=0&serie4=0'
    f'&fecha_desde={FECHA_DESDE}&fecha_hasta={FECHA_HASTA}'
)

ua = UserAgent()

# ── Intento 1: requests + sesión + headers realistas ─────────────────────────
print('=== Intento 1: requests + sesión + headers completos ===')
session = requests.Session()
session.headers.update({
    'User-Agent':                ua.chrome,
    'Accept':                    'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language':           'es-AR,es;q=0.9,en;q=0.8',
    'Accept-Encoding':           'gzip, deflate, br',
    'Referer':                   'https://www.bcra.gob.ar/',
    'Connection':                'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest':            'document',
    'Sec-Fetch-Mode':            'navigate',
    'Sec-Fetch-Site':            'same-origin',
    'DNT':                       '1',
})
# Visitar home primero para obtener cookies de sesión
session.get('https://www.bcra.gob.ar/', verify=False, timeout=15)
respuesta_web = session.get(url_web, verify=False, timeout=15)
print(f'  Status: {respuesta_web.status_code}  {"✅" if respuesta_web.status_code == 200 else "❌"}')

# ── Intento 2: curl_cffi — impersonación TLS de Chrome ───────────────────────
if respuesta_web.status_code != 200:
    print()
    print('=== Intento 2: curl_cffi — impersonación TLS de Chrome ===')
    print('    Replica el handshake SSL/TLS exacto de Chrome (JA3/JA4 fingerprint)')
    respuesta_web = curl_requests.get(
        url_web,
        impersonate='chrome',
        verify=False,
        timeout=15,
    )
    print(f'  Status: {respuesta_web.status_code}  {"✅" if respuesta_web.status_code == 200 else "❌"}')

# ── Intento 3: curl_cffi simulando distintos navegadores ─────────────────────
if respuesta_web.status_code != 200:
    print()
    print('=== Intento 3: curl_cffi — rotar entre navegadores ===')
    for browser in ['chrome110', 'edge99', 'safari15_5', 'firefox']:
        try:
            r = curl_requests.get(url_web, impersonate=browser, verify=False, timeout=10)
            print(f'  [{browser}] Status: {r.status_code}  {"✅" if r.status_code == 200 else "❌"}')
            if r.status_code == 200:
                respuesta_web = r
                break
        except Exception as e:
            print(f'  [{browser}] Error: {e}')

# ── Resultado final ───────────────────────────────────────────────────────────
print()
print(f'Resultado final — Status: {respuesta_web.status_code}')
print(f'Content-Type : {respuesta_web.headers.get("Content-Type", "N/D")}')
print(f'Tamaño HTML  : {len(respuesta_web.text)} caracteres')

if respuesta_web.status_code == 200:
    print('\n✅ Acceso logrado. Primeros 1500 chars:')
    print(respuesta_web.text[:1500])
else:
    print()
    print('⛔ Todos los intentos sin navegador fallaron.')
    print('   Diagnóstico: el BCRA usa un WAF (Web Application Firewall) que')
    print('   bloquea a nivel de infraestructura, independientemente del TLS.')
    print()
    print('   Jerarquía de técnicas de evasión de scraping:')
    print('   ① requests básico            → bloqueado por User-Agent')
    print('   ② headers + sesión           → bloqueado por WAF (intento 1)')
    print('   ③ curl_cffi (TLS spoof)      → bloqueado por WAF (intento 2 y 3)')
    print('   ④ Playwright/Selenium        → ejecuta JS real, pasa WAF')
    print()
    print('   Para este sitio: usar la API REST oficial (Parte B) ✅')


=== Intento 1: requests + sesión + headers completos ===
  Status: 403  ❌

=== Intento 2: curl_cffi — impersonación TLS de Chrome ===
    Replica el handshake SSL/TLS exacto de Chrome (JA3/JA4 fingerprint)
  Status: 403  ❌

=== Intento 3: curl_cffi — rotar entre navegadores ===
  [chrome110] Status: 403  ❌
  [edge99] Status: 403  ❌
  [safari15_5] Status: 403  ❌
  [firefox] Status: 403  ❌

Resultado final — Status: 403
Content-Type : text/html; charset=UTF-8
Tamaño HTML  : 3200 caracteres

⛔ Todos los intentos sin navegador fallaron.
   Diagnóstico: el BCRA usa un WAF (Web Application Firewall) que
   bloquea a nivel de infraestructura, independientemente del TLS.

   Jerarquía de técnicas de evasión de scraping:
   ① requests básico            → bloqueado por User-Agent
   ② headers + sesión           → bloqueado por WAF (intento 1)
   ③ curl_cffi (TLS spoof)      → bloqueado por WAF (intento 2 y 3)
   ④ Playwright/Selenium        → ejecuta JS real, pasa WAF

   Para este sitio: usar l

### A2. Parsear el HTML con BeautifulSoup

In [4]:

if respuesta_web.status_code != 200:
    print(f'⛔ No se puede parsear — status {respuesta_web.status_code}.')
    tablas = []
else:
    soup = BeautifulSoup(respuesta_web.text, 'lxml')
    print('Título de la página:', soup.title.text.strip() if soup.title else 'Sin título')

    tablas = soup.find_all('table')
    print(f'Tablas encontradas: {len(tablas)}')

    if tablas:
        print('\nEncabezados de la primera tabla:')
        encabezados = [th.text.strip() for th in tablas[0].find_all('th')]
        print(encabezados)
        print('\nHTML de la primera tabla (primeras 2000 caracteres):')
        print(tablas[0].prettify()[:2000])
    else:
        # Buscar si los datos vienen en un script JSON embebido
        print('Sin tablas — buscando datos en scripts...')
        scripts = soup.find_all('script')
        print(f'Scripts encontrados: {len(scripts)}')
        for i, script in enumerate(scripts):
            if script.string and ('fecha' in script.string.lower() or 'valor' in script.string.lower()):
                print(f'\nScript {i} contiene datos (primeros 500 chars):')
                print(script.string[:500])
                break


⛔ No se puede parsear — status 403.


### A3. Extraer Datos de la Tabla (si está en el HTML)

In [5]:

filas_scraping = []
df_scraping    = pd.DataFrame()

if respuesta_web.status_code != 200:
    print(f'⛔ Sin datos de scraping — HTTP {respuesta_web.status_code}.')
elif tablas:
    for tabla in tablas:
        for fila in tabla.find_all('tr')[1:]:
            celdas = [td.text.strip() for td in fila.find_all('td')]
            if len(celdas) >= 2:
                filas_scraping.append({'fecha': celdas[0], 'valor': celdas[1]})

    df_scraping = pd.DataFrame(filas_scraping)
    if not df_scraping.empty:
        df_scraping['fecha'] = pd.to_datetime(df_scraping['fecha'], dayfirst=True, errors='coerce')
        df_scraping['valor'] = pd.to_numeric(
            df_scraping['valor'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
            errors='coerce'
        )
        df_scraping['fuente'] = 'scraping'
        print(f'✅ Registros extraídos por scraping: {len(df_scraping)}')
    else:
        print('La tabla está vacía.')
else:
    print('⚠️  Sin tablas en el HTML — los datos pueden cargarse vía JavaScript.')

df_scraping


⛔ Sin datos de scraping — HTTP 403.


""


---
## PARTE B — Enfoque 2: API REST Oficial del BCRA

El BCRA expone una API JSON en `https://api.bcra.gob.ar/estadisticas/v3/`  
Devuelve los mismos datos de forma estructurada, sin necesidad de parsear HTML.

### B1. Petición GET y JSON Crudo

In [9]:

from datetime import date, timedelta

# API activa del BCRA: estadisticascambiarias/v1.0/Cotizaciones
# Devuelve cotizaciones de todas las monedas para una fecha dada
# Iteramos día a día para construir el historial USD

url_api = 'https://api.bcra.gob.ar/estadisticascambiarias/v1.0/Cotizaciones'

# Muestra la respuesta cruda para el día de inicio
respuesta_api = requests.get(url_api, params={'fecha': FECHA_DESDE}, verify=False, timeout=15)

print('URL llamada :', respuesta_api.url)
print('Status code :', respuesta_api.status_code)
print('Content-Type:', respuesta_api.headers.get('Content-Type', 'N/D'))
print('\nJSON raw (primeros 800 caracteres):')
print(respuesta_api.text[:800])


URL llamada : https://api.bcra.gob.ar/estadisticascambiarias/v1.0/Cotizaciones?fecha=2025-04-15
Status code : 200
Content-Type: application/json; charset=utf-8

JSON raw (primeros 800 caracteres):
{"status":200,"results":{"fecha":"2025-04-15","detalle":[{"codigoMoneda":"ARS","descripcion":"PESO","tipoPase":0.00083300,"tipoCotizacion":0.00000000},{"codigoMoneda":"AUD","descripcion":"DOLAR AUSTRALIA","tipoPase":0.63500000,"tipoCotizacion":762.00000000},{"codigoMoneda":"AWG","descripcion":"FLORIN (ANTILLAS HOLANDESAS)","tipoPase":0.55865900,"tipoCotizacion":670.39106100},{"codigoMoneda":"BOB","descripcion":"BOLIVIANOS","tipoPase":0.14505400,"tipoCotizacion":174.06440400},{"codigoMoneda":"BRL","descripcion":"REAL","tipoPase":0.16969600,"tipoCotizacion":203.63488300},{"codigoMoneda":"CAD","descripcion":"DOLAR CANADIEN.","tipoPase":0.71571700,"tipoCotizacion":858.86057800},{"codigoMoneda":"CHF","descripcion":"FRANCO SUIZO","tipoPase":1.21610100,"tipoCotizacion":1459.32141600},{"codigoMone


### B2. Convertir la Respuesta a DataFrame

In [10]:

# Construir historial: iterar cada día hábil desde FECHA_DESDE hasta FECHA_HASTA
# y filtrar la cotización del USD

registros = []
fecha_inicio = date.fromisoformat(FECHA_DESDE)
fecha_fin    = date.fromisoformat(FECHA_HASTA)
cursor       = fecha_inicio

while cursor <= fecha_fin:
    # Solo días hábiles (lun-vie)
    if cursor.weekday() < 5:
        r = requests.get(url_api, params={'fecha': cursor.isoformat()}, verify=False, timeout=15)
        if r.status_code == 200:
            data = r.json()
            for moneda in data.get('results', {}).get('detalle', []):
                if moneda['codigoMoneda'] == 'USD':
                    registros.append({
                        'fecha'         : data['results']['fecha'],
                        'moneda'        : moneda['codigoMoneda'],
                        'descripcion'   : moneda['descripcion'],
                        'tipoCotizacion': moneda['tipoCotizacion']
                    })
    cursor += timedelta(days=1)

df_api = pd.DataFrame(registros)
df_api['fecha'] = pd.to_datetime(df_api['fecha'])

print(f'Registros obtenidos : {len(df_api)}')
print(f'Rango de fechas     : {df_api["fecha"].min().date()} → {df_api["fecha"].max().date()}')
print('\nPrimeras 5 filas:')
print(df_api.head())


Registros obtenidos : 240
Rango de fechas     : 2025-04-15 → 2026-04-15

Primeras 5 filas:
       fecha moneda     descripcion  tipoCotizacion
0 2025-04-15    USD  DOLAR E.E.U.U.          1200.0
1 2025-04-16    USD  DOLAR E.E.U.U.          1135.0
2 2025-04-21    USD  DOLAR E.E.U.U.          1094.0
3 2025-04-22    USD  DOLAR E.E.U.U.          1104.0
4 2025-04-23    USD  DOLAR E.E.U.U.          1160.0


### B3. Estadísticas del Tipo de Cambio

In [12]:

col = 'tipoCotizacion'   # columna con el valor del tipo de cambio

print(f'=== Tipo de Cambio Minorista USD — {FECHA_DESDE} → {FECHA_HASTA} ===')
print(f'  Mínimo  : $ {df_api[col].min():,.2f}  ({df_api.loc[df_api[col].idxmin(), "fecha"].date()})')
print(f'  Máximo  : $ {df_api[col].max():,.2f}  ({df_api.loc[df_api[col].idxmax(), "fecha"].date()})')
print(f'  Promedio: $ {df_api[col].mean():,.2f}')
print(f'  Último  : $ {df_api.iloc[-1][col]:,.2f}  ({df_api.iloc[-1]["fecha"].date()})')
print(f'  Registros: {len(df_api)} días hábiles')

# Resumen mensual
df_api['mes'] = df_api['fecha'].dt.to_period('M')
resumen_mensual = df_api.groupby('mes')[col].agg(['mean', 'min', 'max']).round(2)
resumen_mensual.columns = ['Promedio', 'Mínimo', 'Máximo']
print('\nPromedio mensual ($ por USD):')
print(resumen_mensual.to_string())


=== Tipo de Cambio Minorista USD — 2025-04-15 → 2026-04-15 ===
  Mínimo  : $ 1,094.00  (2025-04-21)
  Máximo  : $ 1,492.00  (2025-10-24)
  Promedio: $ 1,346.57
  Último  : $ 1,359.00  (2026-04-15)
  Registros: 240 días hábiles

Promedio mensual ($ por USD):
         Promedio  Mínimo  Máximo
mes                              
2025-04   1154.95  1094.0  1200.0
2025-05   1148.30  1114.0  1200.0
2025-06   1180.74  1142.5  1205.0
2025-07   1269.80  1222.0  1374.0
2025-08   1328.25  1292.5  1364.0
2025-09   1401.07  1326.0  1475.0
2025-10   1435.09  1349.0  1492.0
2025-11   1428.15  1387.0  1482.0
2025-12   1447.95  1435.0  1457.0
2026-01   1448.95  1429.5  1475.0
2026-02   1408.39  1370.5  1451.0
2026-03   1395.95  1368.0  1416.0
2026-04   1377.22  1354.0  1394.0


---
## PARTE C — Comparación: Scraping vs API

In [ ]:
comparacion = pd.DataFrame([
    {
        'Criterio':         'Datos obtenidos',
        'Web Scraping':     f'{len(df_scraping)} registros' if not df_scraping.empty else '0 (carga dinámica JS)',
        'API REST':         f'{len(df_api)} registros',
    },
    {
        'Criterio':         'Formato de respuesta',
        'Web Scraping':     'HTML — requiere parseo',
        'API REST':         'JSON — estructurado directamente',
    },
    {
        'Criterio':         'Fragilidad',
        'Web Scraping':     'Alta — rompe si cambia el diseño web',
        'API REST':         'Baja — contrato de interfaz estable',
    },
    {
        'Criterio':         'Contenido dinámico (JS)',
        'Web Scraping':     'No capturado (necesita Selenium)',
        'API REST':         'No aplica — datos directos',
    },
    {
        'Criterio':         'Autenticación requerida',
        'Web Scraping':     'No',
        'API REST':         'No (BCRA API pública)',
    },
    {
        'Criterio':         'Código necesario',
        'Web Scraping':     'requests + BeautifulSoup + lógica de parseo',
        'API REST':         'requests + .json()',
    },
    {
        'Criterio':         'Recomendado',
        'Web Scraping':     'Solo si no hay API',
        'API REST':         '✅ Siempre que esté disponible',
    },
])

comparacion.set_index('Criterio')

---
## PARTE D — Guardar Resultados en CSV y Parquet

In [ ]:
# Guardar datos de la API
df_api.drop(columns='mes', errors='ignore').to_csv(
    DIR_OUTPUT / f'bcra_serie{SERIE}_{FECHA_DESDE}_{FECHA_HASTA}.csv',
    index=False, encoding='utf-8-sig'
)
df_api.drop(columns='mes', errors='ignore').to_parquet(
    DIR_OUTPUT / f'bcra_serie{SERIE}_{FECHA_DESDE}_{FECHA_HASTA}.parquet',
    index=False, engine='pyarrow'
)

print(f'✅ CSV     : bcra_serie{SERIE}_{FECHA_DESDE}_{FECHA_HASTA}.csv')
print(f'✅ Parquet : bcra_serie{SERIE}_{FECHA_DESDE}_{FECHA_HASTA}.parquet')

# Guardar scraping si tiene datos
if not df_scraping.empty:
    df_scraping.to_csv(DIR_OUTPUT / f'bcra_scraping_serie{SERIE}.csv', index=False, encoding='utf-8-sig')
    print(f'✅ CSV scraping: bcra_scraping_serie{SERIE}.csv')
else:
    print('ℹ️  Sin datos de scraping (página usa carga dinámica).')

# Resumen de archivos
archivos = [
    {'archivo': f.name, 'formato': f.suffix, 'tamaño_KB': round(f.stat().st_size / 1024, 2)}
    for f in sorted(DIR_OUTPUT.iterdir())
]
print('\nArchivos en datos/output:')
pd.DataFrame(archivos)